In [47]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy import stats

In [48]:

# AUD/USD: weekly, January 1994 to December 2003
fx = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXUSAL",
                 index_col=0, parse_dates=True).iloc[:, 0]
fx = pd.to_numeric(fx, errors="coerce").dropna()["1994-01-01":"2003-12-31"]
fx_weekly = fx.resample("W-TUE").last().dropna()
aud_returns = np.log(fx_weekly).diff().dropna()

# DAX: daily, 3 January 2000 to 17 May 2001
dax = yf.download("^GDAXI", start="2000-01-03", end="2001-05-18")["Close"].squeeze()
dax_returns = np.log(dax).diff().dropna()

print(len(aud_returns), len(dax_returns))

[*********************100%***********************]  1 of 1 completed

522 348


### There are 95 values that are NA for the FX data (I dropped them)

Il parle de 529 observation pour le fx (j'en ai 521) et de 1000 pour le dax j'en ai 349 mais c'est impossible d'avoir 1000 daily en 1ans et demie...

In [49]:
def describe(r):
    print("n:", len(r))
    print("mean:", r.mean())
    print("std:", r.std())
    print("skew:", stats.skew(r))
    print("kurtosis:", stats.kurtosis(r, fisher=False))

describe(aud_returns)
describe(dax_returns)

n: 522
mean: 0.0001801679678569659
std: 0.013130348452051658
skew: 0.10262523340696203
kurtosis: 3.604342338762955
n: 348
mean: -0.00025672096418956044
std: 0.01527718605982339
skew: 0.020472582098772694
kurtosis: 2.944466821130261


We standardize the data and find the closest kurt and skew from the paper data for the AUD/USD

In [50]:
days=["W-MON","W-TUE","W-WED","W-THU","W-FRI"]
for day in days:
    fx_weekly = fx.resample(day).last().dropna()
    aud_returns = np.log(fx_weekly).diff().dropna()
    aud = (aud_returns - aud_returns.mean()) / aud_returns.std()
    print(day)
    describe(aud)
    print("-"*30)

W-MON
n: 522
mean: 6.805964901916668e-18
std: 1.0
skew: -0.14781555923278636
kurtosis: 3.196846705595866
------------------------------
W-TUE
n: 522
mean: -1.3611929803833336e-17
std: 1.0
skew: 0.10262523340696203
kurtosis: 3.6043423387629554
------------------------------
W-WED
n: 521
mean: -6.8190281742811916e-18
std: 1.0
skew: -0.18206480699933392
kurtosis: 3.5260750625508654
------------------------------
W-THU
n: 521
mean: -1.3638056348562383e-17
std: 1.0
skew: -0.2530813702627039
kurtosis: 4.115981405360435
------------------------------
W-FRI
n: 521
mean: 0.0
std: 1.0
skew: -0.2806876652411246
kurtosis: 4.817041953470401
------------------------------


Given mean, std.dev, skew and kurtosis :

Australia/US
Mean	−0.04 \
Std.dev.	1.00 \
Skewness	0.06 \
Kurtosis	3.37 

The one for AUD/USD that correspond the most to the data is if we take the weekly data starting from the Tuesday with a skew of 0.1 and a kurt of 3.6

For the DAX the mean, std and kurt are similar but the skew is different 0.2 vs 0.02 (maybe a typo)

In [51]:
fx_weekly = fx.resample("W-TUE").last().dropna()
aud_returns = np.log(fx_weekly).diff().dropna()
aud = (aud_returns - aud_returns.mean()) / aud_returns.std()

In [ ]:
def simulate_sv(n_obs, beta, phi, sigma, seed=0):
    """_summary_

    Args:
        n_obs (_type_): Number of observation
        beta (_type_): beta
        phi (_type_): phi
        sigma (_type_): sigma
        seed (int, optional): seed to have reproductible results

    Returns:
        _type_: generate data from the given parameters
    """
    rng = np.random.default_rng(seed)
    h = np.zeros(n_obs)
    h[0] = rng.normal(0, sigma / np.sqrt(1 - phi**2)) #Variance from Eq10 and the mean is 0 because we use from eq9 ℎ_𝑡 ⁡= 𝜙*⁢ℎ_𝑡−1 + 𝜂_𝑡
    for t in range(1, n_obs):
        h[t] = phi * h[t-1] + sigma * rng.normal() # AR(1) formula with 𝜂_𝑡 = N(0,1) 𝜂_𝑡 = σ_𝜂 * N(0,1)
    y = beta * np.exp(h / 2) * rng.normal(size=n_obs) # 𝑦𝑡⁢=𝛽⁢exp⁡(ℎ𝑡⁡/2)⁢𝜖𝑡  with 𝜖𝑡=N(0,1)
    return y, h

#BPF, still need the APF
#Look at Appendix A.3 Algorithm A1 and 3.2.1 of the paper
def bootstrap_pf(y, beta, phi, sigma, N, c=2, seed=None):
    rng = np.random.default_rng(seed)
    h = rng.normal(0, sigma / np.sqrt(1 - phi**2), N) # (A1) Generate N particles from the stationary distribution, the same one used for h_1 in the simulator.
    log_W = np.full(N, -np.log(N)) # W_0^(i) = 1/N
    log_L_hat = 0.0
    for t in range(len(y)):
        if t > 0:
            h = phi * h + sigma * rng.normal(size=N) # (A2) Eq20 = N(φ * h_{t-1}^(i), σ_η²)
            
        log_w = (-0.5 * np.log(2 * np.pi) - np.log(beta) - h / 2
                - y[t]**2 / (2 * beta**2 * np.exp(h)))  # (A3) w_t^(i) = g * f / q, and with q = f this is w_t^(i) = g(y_t | h_t^(i)) where # y_t | h_t ~ N(0, β² * e^{h_t})
                                                        # log g(y_t | h_t) g is the density functino of yt|ht
                                                        
         # (A4) p_hat(y_t | y_{1:t-1}) = sum_i W_{t-1}^(i) * w_t^(i)
        log_Ww = log_w + log_W # w_t = omega_{t-1} * g
        m = log_Ww.max()
        log_p_hat = m + np.log(np.sum(np.exp(log_Ww - m)))
        log_L_hat += log_p_hat
        
        # (A5)  W_t^(i) proportional to W_{t-1}^(i) * w_t^(i), normalised to sum to 1
        log_W = log_Ww - log_p_hat
        W = np.exp(log_W)
        
        # (A6) resample if N_eff = 1 / sum_i (W_t^(i))^2 < N / c   Eq22
        N_eff = 1.0 / np.sum(W**2)
        if N_eff < N / c:                                     
            h = h[rng.choice(N, size=N, p=W)]
            log_W = np.full(N, -np.log(N)) # weights reset to 1/N
    return log_L_hat

In [60]:
y, h = simulate_sv(n_obs=500, beta=0.9, phi=0.95, sigma=0.2, seed=1)

for N in [100, 1000, 10000]:
    loglik = [bootstrap_pf(y, 0.9, 0.95, 0.2, N=N, seed=s) for s in range(20)]
    print(N, np.mean(loglik), np.std(loglik))

100 -670.5118724277711 0.8981306467581488
1000 -670.1290526939346 0.2004221092890045
10000 -670.0649848098063 0.11176453561667148


So Above we simulate data that will then be use to see if our models work and retrieve the right parameters. To then be able to use this models on the real data to find the parameters ($\beta$, $\phi$ and $\sigma_\eta$)